# DEMO: Metric Views

**Metric Views** in Databricks provide a centralized way to define and manage business metrics by separating measure definitions from the fields used to group, filter, and aggregate them:
- Define measures (aggregations) and fields (grouping/filtering columns) **once** in Unity Catalog
- Query with **any combination** of fields at runtime
- Ensures **consistent metric definitions** across all dashboards and tools

In this demo, you'll:
1. Create a metric view using the **Catalog Explorer UI** (no code required)
2. Query it with SQL in the **SQL Editor** using `MEASURE()` syntax

> **Note:** Run the setup cell below to create the source table, then follow the UI instructions to create the metric view in Catalog Explorer.

---

In [0]:
%run ./Setup

## Why Not Just Use a Normal View?

A common first question: *"Why do I need a metric view? Can't I just write a normal view with `SUM(revenue)`?"*

Yes — but you'd need a **separate view for every combination of grouping** you want:

```sql
-- Normal views: one fixed grouping each
CREATE VIEW revenue_by_region AS SELECT region, SUM(revenue) FROM orders GROUP BY region;
CREATE VIEW revenue_by_month AS SELECT month, SUM(revenue) FROM orders GROUP BY month;
CREATE VIEW revenue_by_region_and_month AS SELECT region, month, SUM(revenue) FROM orders GROUP BY region, month;
-- ... and so on for every combination
```

A **metric view** solves this by separating the *what* (the aggregation) from the *how* (the grouping):

```sql
-- One metric view, unlimited groupings at query time:
SELECT `Region`, MEASURE(`Total Revenue`) FROM order_metrics GROUP BY ALL;
SELECT `Order Month`, MEASURE(`Total Revenue`) FROM order_metrics GROUP BY ALL;
SELECT `Region`, `Order Month`, MEASURE(`Total Revenue`) FROM order_metrics GROUP BY ALL;
```

**Think of it like a spreadsheet pivot table:**
- The **measures** are the values in the pivot (e.g., SUM of Revenue)
- The **fields** are the rows/columns/filters you can drag in and out freely
- You define the measures once, then slice and dice however you like at query time

---

## Part 1: Creating a Metric View

A metric view has three core components:

1. **Source** — The table, view, or SQL query the metrics are calculated from
2. **Fields** — Columns that behave like regular table columns: categorical attributes for grouping and filtering (e.g., region, order month) or unaggregated numeric columns (e.g., price, quantity)
3. **Measures** — Aggregate expressions that produce business metrics (e.g., `SUM(revenue)`, `COUNT(order_id)`)

The definition uses **YAML** syntax inside a `CREATE VIEW ... WITH METRICS` statement.

**Key difference from standard views:**
- A standard view locks in the aggregation and grouping at creation time
- A metric view defines WHAT to aggregate but lets you choose HOW to group at query time

> **Note:** The `dimensions` keyword is accepted as a backward-compatible synonym for `fields` in YAML definitions, but the current terminology is **fields**.

---

## Step-by-Step: Create the Metric View in Catalog Explorer

### Step 1: Navigate to your source table
1. Click **Catalog** in the left sidebar
2. Browse to your schema: `workspace` > `demo_metric_views_<your_username>` > `gold_orders`
3. Click the **gold_orders** table name to open the table details page

### Step 2: Create the metric view
1. Click the **Create** button (top right of the table details page)
2. Select **Metric view** from the dropdown
3. In the dialog:
   - **Name:** `order_metrics`
   - **Catalog:** `workspace`
   - **Schema:** `demo_metric_views_<your_username>`
4. Click **Create**

### Step 3: Define fields
The low-code UI editor opens. Click **UI** to open the low-code editor if necessary. All source columns are added as fields automatically.

1. Review the auto-added fields (region, product_category, customer_segment, channel, order_date)
2. Optionally add a **calculated field** by clicking **+ Add** under Fields:
   - Field name: `order_month`
   - Expression: `DATE_TRUNC('month', order_date)`
   - Display name: `Order Month`
   - Comment: `Month the order was placed`
3. Click **Save** in the top right

### Step 4: Define measures
Measures are aggregate expressions that produce business metrics. Click **+ Add** under Measures to add each one:

| Measure Name | Display Name | Expression | Comment |
| --- | --- | --- | --- |
| total_revenue | Total Revenue | `SUM(revenue)` | Sum of all order revenue |
| total_cost | Total Cost | `SUM(cost)` | Sum of all order costs |
| order_count | Order Count | `COUNT(order_id)` | Number of orders |
| total_quantity | Total Quantity | `SUM(quantity)` | Total units sold |
| avg_order_value | Avg Order Value | `SUM(revenue) / COUNT(order_id)` | Average revenue per order |
| profit_margin_pct | Profit Margin Pct | `(SUM(revenue) - SUM(cost)) / SUM(revenue) * 100` | Profit margin as percentage |

For each measure, you can optionally:
- Click **Preview** next to the measure name to see a sample result
- Click **Generate with AI** to use Genie Code to suggest a measure expression
- Add a **Display name**, **Comment**, **Synonyms**, **Format** (e.g., Currency USD for revenue), or **Tags**

### Step 5: Save
Click **Save** at the top of the editor.

Your metric view is now live in Unity Catalog and queryable from any SQL tool!

---

> **Alternative:** You can also switch to the **YAML** tab in the editor to see/edit the raw YAML definition directly. This is useful for version control or complex definitions. The YAML editor shows the full definition including `fields:` and `measures:` sections.

## Part 2: Querying a Metric View with MEASURE()

Querying a metric view is different from querying a regular table. The key rules:

| Rule | Example |
| --- | --- |
| Reference fields directly by name | `order_month` — no wrapper needed |
| Wrap measures with `MEASURE()` | `MEASURE(total_revenue)` |
| Always use `GROUP BY ALL` | Required when selecting measures |
| Never use `SELECT *` | Must explicitly list fields and measures |
| Backtick-escape names with spaces/special chars | `order_date`, `total_revenue` |

**Example combining fields and measures:**
```sql
SELECT
  `Region`,                          -- field: just use the name
  `Order Month`,                     -- field: just use the name
  MEASURE(`Total Revenue`) AS rev,   -- measure: must wrap with MEASURE()
  MEASURE(`Order Count`) AS orders   -- measure: must wrap with MEASURE()
FROM order_metrics
GROUP BY ALL;
```

> **Tip:** In Databricks Runtime 18.1+, `AGG()` is an interchangeable alias for `MEASURE()` — the same rules apply to both.

> **Now open the SQL Editor** (left sidebar) to run the queries below. Make sure to set your catalog and schema correctly first.

---

In [0]:
%sql
-- ============================================================
-- Query 1: Revenue by Region (simplest query)
-- ============================================================

SELECT
  `Region`,
  MEASURE(total_revenue) AS `Total Revenue`
FROM order_metrics
GROUP BY ALL
ORDER BY `Total Revenue` DESC;

In [0]:
%sql
-- ============================================================
-- Query 2: Multiple measures by Category
-- ============================================================

SELECT
  `Product Category`,
  MEASURE(total_revenue) AS `Total Revenue`,
  MEASURE(order_count) AS `Order Count`,
  MEASURE(avg_order_value) AS `Avg Order Value`,
  MEASURE(profit_margin_pct) AS `Profit Margin Pct`
FROM order_metrics
GROUP BY ALL
ORDER BY `Total Revenue` DESC;

In [0]:
%sql
-- ============================================================
-- Query 3: Monthly revenue trend
-- ============================================================

SELECT
  MEASURE(total_revenue) AS `Total Revenue`,
  MEASURE(order_count) AS `Order Count`
FROM order_metrics
GROUP BY ALL
ORDER BY order_month ASC;

In [0]:
%sql
-- ============================================================
-- Query 4: Filter by field (like a Power BI slicer)
-- ============================================================

SELECT
  `Product Category`,
  `Channel`,
  MEASURE(total_revenue) AS `Total Revenue`,
  MEASURE(profit_margin_pct) AS `Profit Margin Pct`
FROM order_metrics
WHERE `Region` = 'Northeast'
  AND `Order Date` >= '2024-07-01'
GROUP BY ALL
ORDER BY `Total Revenue` DESC;

In [0]:
%sql
-- ============================================================
-- Query 5: Filter by measure value (HAVING)
-- ============================================================
-- Note: HAVING filters on the alias, not MEASURE() directly.

SELECT
  `Customer Segment`,
  MEASURE(total_revenue) AS `Total Revenue`,
  MEASURE(order_count) AS `Order Count`
FROM order_metrics
GROUP BY ALL
HAVING `Total Revenue` > 50000
ORDER BY `Total Revenue` DESC;

## Part 3: Why This Matters — Define Once, Use Everywhere

The power of metric views is **consistency**:

| Without Metric Views | With Metric Views |
| --- | --- |
| Dashboard A calculates revenue as `SUM(amount)` | All dashboards use `MEASURE(total_revenue)` |
| Dashboard B calculates revenue as `SUM(price * qty)` | Same definition everywhere |
| Numbers disagree → trust erodes | Numbers always match → single source of truth |

**Other benefits:**
- Stored in **Unity Catalog** — governed, versioned, discoverable
- Any team member can query the same metrics with any combination of fields
- When the definition changes (e.g., exclude returns), it updates everywhere
- AI/BI dashboards and Genie can discover and use metric view definitions
- Use `synonyms` to improve AI discoverability of fields and measures

---

In [0]:
%sql
-- ============================================================
-- Query 6: Grand totals (no fields = overall summary)
-- ============================================================

SELECT
  MEASURE(total_revenue) AS `Total Revenue`,
  MEASURE(total_cost) AS `Total Cost`,
  MEASURE(order_count) AS `Order Count`,
  MEASURE(avg_order_value) AS `Avg Order Value`,
  MEASURE(profit_margin_pct) AS `Profit Margin Pct`
FROM order_metrics
GROUP BY ALL;

---

## Summary

| Concept | Key Takeaway |
| --- | --- |
| **Metric View** | A Unity Catalog object that defines reusable measures + fields |
| **MEASURE()** / **AGG()** | Required function wrapper to evaluate a defined measure (AGG is an alias in DBR 18.1+) |
| **GROUP BY ALL** | Always required when querying measures |
| **Fields** | Choose any combination at query time for grouping/filtering (like Power BI slicer/rows/columns) |
| **Consistency** | Same metric definition used across all dashboards — single source of truth |


**When to use metric views:**
- A metric is used across **multiple dashboards** (revenue, order count, conversion rate)
- You want a **single source of truth** for business KPIs
- Multiple teams need to query the **same metrics** with different groupings
- You want AI/BI tools to discover and use standardized metric definitions